# DCS Constraints Extraction & Coverage for 03LIC_1071 Control-Action Tags

**Goal:** Read the green-flagged DCS constraint parameters from `DCS_Constraints_For_RCA_CA.xlsx`, extract their values for every control-action tag used in 03LIC_1071 alarm episodes from the Honeywell `Tags_Parameters.csv` dump, and report a coverage matrix.

**Critical data facts:**
- The Excel table has an empty first row/column; headers are in row 2, data from row 3.
- In `Tags_Parameters.csv`, numeric limits live on function blocks where `StrategyName` is **blank**; the tag is identified only by the block's `NAME` row.
- Controllers are either `PID` (LIC/PIC/FIC/TIC — have OP + SP + PV limits) or `AUTOMAN` (HIC hand controllers — OP limits only).
- Values are coalesced from `IntergerValue` / `RealValue` / `StringValue`.
- `1.7899999123456789E308` means "unbounded / disabled".

**Outputs:**
- `RESULTS/dcs_constraints_1071/dcs_constraints_coverage_1071.xlsx`


In [12]:
# ---- imports + paths ---------------------------------------------------------
import pandas as pd
import numpy as np
import os, re, glob
import openpyxl
from collections import defaultdict, Counter

DATA    = '/home/h604827/ControlActions/DATA'
RESULTS = '/home/h604827/ControlActions/RESULTS'

XLS_CONSTRAINTS = f'{DATA}/Constraints/DCS_Constraints_For_RCA_CA.xlsx'
CSV_DIR         = f'{DATA}/Constraints/Tags_Parameters.csv'
ACTIONS_XLS     = f'{RESULTS}/03LIC_1071_PVLO_episodes_12JUN2026_1219/03LIC_1071_pvlo_alarms_clustered_with_control_actions.xlsx'
OUT_DIR         = f'{RESULTS}/dcs_constraints_1071'
OUT_XLS         = f'{OUT_DIR}/dcs_constraints_coverage_1071.xlsx'

os.makedirs(OUT_DIR, exist_ok=True)
print('paths configured')

paths configured


## Phase A — Extract green parameters from Excel

Locate the `Parameter List` column, inspect fill colors, and collect the rows whose background is green. The Excel uses theme colors; the green rows have `theme=9, tint=0.0`.

In [13]:
# ---- parse Excel colors and green rows ---------------------------------------
wb = openpyxl.load_workbook(XLS_CONSTRAINTS, data_only=False)
ws = wb.active
print('Sheet:', ws.title, '| rows:', ws.max_row, '| cols:', ws.max_column)

# header is row 2; first col is empty
headers = [ws.cell(row=2, column=c).value for c in range(1, ws.max_column+1)]
print('Headers row 2:', headers)

pl_col = next((i for i, h in enumerate(headers, 1)
               if h and 'parameter' in str(h).lower() and 'list' in str(h).lower()), None)
assert pl_col, 'Parameter List column not found'
print('Parameter List column index:', pl_col)

rows_info = []
for r in range(3, ws.max_row+1):
    cell = ws.cell(row=r, column=pl_col)
    val = cell.value
    fg = cell.fill.fgColor
    rgb = fg.rgb if fg and fg.rgb and fg.rgb != '00000000' else None
    # theme/tint may be openpyxl descriptor objects or plain ints; convert safely
    def _to_int(x):
        try:
            return int(str(x))
        except Exception:
            return None
    def _to_float(x):
        try:
            return float(str(x))
        except Exception:
            return None
    theme = _to_int(fg.theme) if fg and fg.theme is not None else None
    tint = _to_float(fg.tint) if fg and fg.tint is not None else None
    rows_info.append({'row': r, 'value': val, 'rgb': rgb, 'theme': theme, 'tint': tint})

df_color = pd.DataFrame(rows_info)
print('\nColor (rgb,theme,tint) -> count:')
print(df_color.groupby(['rgb','theme','tint'], dropna=False).size().sort_values(ascending=False))

# Theme 9 tint 0.0 is the green in this workbook
green_mask = (df_color['theme'] == 9) & (df_color['tint'] == 0.0)
green_raw = df_color.loc[green_mask, 'value'].dropna().tolist()
print('\nGreen raw Parameter List values:')
for v in green_raw:
    print(' ', repr(v))

# Explode compound entries like "PVHITP / PVLOTP \nPVHIPR / PVLOPR"
def explode_params(raw):
    parts = re.split(r'[\/\
,]+', str(raw))
    return [p.strip().replace(' ', '') for p in parts if p.strip()]

green_params = []
for raw in green_raw:
    green_params.extend(explode_params(raw))
green_params = list(dict.fromkeys(green_params))  # ordered dedup
print('\nExploded green parameters ({}):'.format(len(green_params)))
print(green_params)

# Assign simple families
OP_FAMILY = {'OPHITP','OPLOTP','OPHILM','OPLOLM','OPROCLM','OPMCHLM','OPALDB','$OPTOL'}
SP_FAMILY = {'SPHILM','SPLOLM','SPTOL'}
PV_FAMILY = {'PVHITP','PVLOTP','PVHHTP','PVLLTP','PVROCPTP','PVROCNTP','PVEUHI','PVEULO',
             'PVEXEUHI','PVEXEULO','PVHIPR','PVLOPR','CVEULO','CVEUHI','DEVHITP','DEVLOTP'}
OTHER_FAMILY = {'CTLACTN','PNTFORM','NMODE','PVLOFL','PVHIFL','REDTAG','PVSOURCE','OROPT',
                'SHEDMODE','SHEDTIME'}

def family_of(p):
    p = p.upper()
    if p in OP_FAMILY: return 'OP'
    if p in SP_FAMILY: return 'SP'
    if p in PV_FAMILY: return 'PV'
    return 'OTHER'

df_green = pd.DataFrame({
    'ParameterName': green_params,
    'raw_ParameterList': [next((r for r in green_raw if p in explode_params(r)), '') for p in green_params],
    'family': [family_of(p) for p in green_params]
})
display(df_green)

Sheet: DCS Parameters | rows: 26 | cols: 4
Headers row 2: [None, 'Parameter List', 'Description', 'Comments']
Parameter List column index: 2

Color (rgb,theme,tint) -> count:
rgb                                   theme  tint     
Values must be of type <class 'str'>  9.0     0.000000    9
                                      2.0    -0.249977    8
NaN                                   NaN     0.000000    4
FFFF3300                              NaN     0.000000    3
dtype: int64

Green raw Parameter List values:
  'PVHITP / PVLOTP \nPVHIPR / PVLOPR'
  'OPHITP / OPLOTP'
  'SPHILM / SPLOLM'
  'CTLACTN'
  'SPTOL'
  '$OPTOL'
  'OPROCLM'
  'PVROCPTP / PVROCNTP'

Exploded green parameters (14):
['PVHITP', 'PVLOTP', 'PVHIPR', 'PVLOPR', 'OPHITP', 'OPLOTP', 'SPHILM', 'SPLOLM', 'CTLACTN', 'SPTOL', '$OPTOL', 'OPROCLM', 'PVROCPTP', 'PVROCNTP']


,ParameterName,raw_ParameterList,family
0,PVHITP,PVHITP / PVLOTP \nPVHIPR / PVLOPR,PV
1,PVLOTP,PVHITP / PVLOTP \nPVHIPR / PVLOPR,PV
2,PVHIPR,PVHITP / PVLOTP \nPVHIPR / PVLOPR,PV
3,PVLOPR,PVHITP / PVLOTP \nPVHIPR / PVLOPR,PV
4,OPHITP,OPHITP / OPLOTP,OP
5,OPLOTP,OPHITP / OPLOTP,OP
6,SPHILM,SPHILM / SPLOLM,SP
7,SPLOLM,SPHILM / SPLOLM,SP
8,CTLACTN,CTLACTN,OTHER
9,SPTOL,SPTOL,SP


## Phase B — Control-action tags from the 1071 workbook

The `control_actions` sheet lists every operator action. `Source` is the base tag; `Description` tells us whether the action changed OP, SP, MODE, etc. We focus on `.OP` and `.SP` handles.

In [14]:
# ---- load control actions and enumerate tags/handles -------------------------
df_actions = pd.read_excel(ACTIONS_XLS, sheet_name='control_actions')
print('Control actions shape:', df_actions.shape)
print('Columns:', df_actions.columns.tolist())

source_tags = sorted(df_actions['Source'].dropna().unique())
print('\nUnique Source tags:', len(source_tags))

# action kinds per tag
kind_by_tag = (df_actions.groupby('Source')['Description']
               .apply(lambda s: sorted(s.dropna().unique().tolist()))
               .to_dict())

handles = []
for tag in source_tags:
    kinds = kind_by_tag.get(tag, [])
    if 'OP' in kinds:
        handles.append((tag, 'OP'))
    if 'SP' in kinds:
        handles.append((tag, 'SP'))
print('OP/SP handles:', len(handles))

# Focus handles requested by user
focus = ['03LIC_1071.OP', '03PIC_1023.OP', '03LIC_1034.SP', '03HIC_1151.OP', '03LIC_1016.OP']
# normalize missing underscore
focus = [re.sub(r'03HIC(\d)', r'03HIC_', h) for h in focus]
print('\nFocus handles:', focus)
missing_focus = [h for h in focus if tuple(h.split('.')) not in handles]
if missing_focus:
    print('WARNING: focus handles not found in actions:', missing_focus)

pd.DataFrame(handles, columns=['tag','kind']).head(10)

Control actions shape: (42290, 12)
Columns: ['cluster_id', 'cluster_start', 'cluster_end', 'action_timing', 'action_direction', 'Source', 'Description', 'VT_Start', 'PrevValue', 'Value', 'source_table', 'filtered']

Unique Source tags: 195
OP/SP handles: 179

Focus handles: ['03LIC_1071.OP', '03PIC_1023.OP', '03LIC_1034.SP', '03HIC_1151.OP', '03LIC_1016.OP']


,tag,kind
0,02EDPV_1052,OP
1,02EDPV_1094,OP
2,02FAX_1247,OP
3,02FIC_1247,OP
4,02HIC_1050,OP
5,02HIC_1087,OP
6,02KM_1139_ST,OP
7,02LI_1041_MOS,OP
8,02LI_1085_MOS,OP
9,02PDC_1220_MAN,OP


## Phase C — Extract constraint values from Tags_Parameters.csv

The CSV contains named AI points (`StrategyName` present) and controller blocks (`StrategyName` blank, tag in `NAME` row). We segment blocks by anchors (non-empty StrategyName or `ParameterName=='NAME'`), forward-fill the tag, keep only green parameters, and coalesce the value across the three value columns.

In [15]:
# ---- load and segment the DCS parameter dump ---------------------------------
part_files = sorted(glob.glob(f'{CSV_DIR}/part-*.csv'))
part_files = [f for f in part_files if f.endswith('.csv') and ':Zone.Identifier' not in f
              and ':sec.endpointdlp' not in f]
print('CSV part files:', part_files)

df_csv = pd.concat(
    [pd.read_csv(f, dtype=str, keep_default_na=False, na_filter=False) for f in part_files],
    ignore_index=True
)
print('CSV rows:', len(df_csv), '| cols:', df_csv.columns.tolist())

# block segmentation
is_anchor = (df_csv['StrategyName'] != '') | (df_csv['ParameterName'] == 'NAME')
df_csv['anchor_tag'] = np.where(
    df_csv['StrategyName'] != '', df_csv['StrategyName'],
    np.where(df_csv['ParameterName'] == 'NAME', df_csv['StringValue'], np.nan)
)
df_csv['effective_tag'] = df_csv['anchor_tag'].ffill()
df_csv['block_template'] = pd.Series(np.where(is_anchor, df_csv['TemplateName'], np.nan)).ffill()

# coalesce value
val_cols = ['IntergerValue','RealValue','StringValue']

def first_non_empty(row):
    for c in val_cols:
        v = row[c]
        if pd.notna(v) and str(v).strip() != '':
            return str(v).strip(), c
    return '', ''

df_csv[['raw_value','value_col']] = df_csv.apply(first_non_empty, axis=1, result_type='expand')

# interpret unbounded sentinel
SENTINEL = '1.7899999123456789E308'
def interpret_value(v):
    if v == SENTINEL:
        return 'UNBOUNDED'
    return v

# keep green parameters only
df_green_vals = df_csv[df_csv['ParameterName'].isin(green_params)].copy()
df_green_vals['interpreted_value'] = df_green_vals['raw_value'].apply(interpret_value)
df_green_vals = df_green_vals[['effective_tag','block_template','ParameterName',
                                'raw_value','interpreted_value','value_col']].rename(
    columns={'effective_tag':'tag','block_template':'template'})
print('Green-parameter rows found:', len(df_green_vals))
print('Unique tags with green params:', df_green_vals['tag'].nunique())
display(df_green_vals.head(20))

CSV part files: ['/home/h604827/ControlActions/DATA/Constraints/Tags_Parameters.csv/part-00000-76dde257-7d3f-4849-acbb-5e8e58c698f1-c000.csv', '/home/h604827/ControlActions/DATA/Constraints/Tags_Parameters.csv/part-00001-76dde257-7d3f-4849-acbb-5e8e58c698f1-c000.csv', '/home/h604827/ControlActions/DATA/Constraints/Tags_Parameters.csv/part-00002-76dde257-7d3f-4849-acbb-5e8e58c698f1-c000.csv', '/home/h604827/ControlActions/DATA/Constraints/Tags_Parameters.csv/part-00003-76dde257-7d3f-4849-acbb-5e8e58c698f1-c000.csv']
CSV rows: 764814 | cols: ['StrategyName', 'ParameterName', 'IntergerValue', 'RealValue', 'StringValue', 'TemplateName']
Green-parameter rows found: 17712
Unique tags with green params: 2963


,tag,template,ParameterName,raw_value,interpreted_value,value_col
170,03LY_2272,,$OPTOL,,,
532,03TI_2011BA2,,PVHIPR,NOACTION,NOACTION,StringValue
536,03TI_2011BA2,,PVLOPR,NOACTION,NOACTION,StringValue
561,03TI_2011BA2,,PVHITP,,,
562,03TI_2011BA2,,PVLOTP,,,
565,03TI_2011BA2,,PVROCPTP,,,
566,03TI_2011BA2,,PVROCNTP,,,
1508,02PI_1245,,PVHIPR,NOACTION,NOACTION,StringValue
1512,02PI_1245,,PVLOPR,LOW,LOW,StringValue
1537,02PI_1245,,PVHITP,,,


### Spot-check: 03LIC_1071 PID block values

These should match the known DCS limits for the target tag.

In [16]:
check = df_green_vals[df_green_vals['tag'] == '03LIC_1071'].sort_values('ParameterName')
display(check)
assert set(check['ParameterName']) >= {'PVLOTP','PVHITP','PVHIPR','PVLOPR','SPHILM','SPLOLM'},     'Expected green PID params not found for 03LIC_1071'
assert check.set_index('ParameterName').loc['PVLOTP','interpreted_value'] == '28.75', 'PVLOTP mismatch'
assert check.set_index('ParameterName').loc['PVHITP','interpreted_value'] == '71.25', 'PVHITP mismatch'
print('03LIC_1071 spot-check OK')

,tag,template,ParameterName,raw_value,interpreted_value,value_col
705489,03LIC_1071,PID,$OPTOL,,,
705511,03LIC_1071,PID,CTLACTN,REVERSE,REVERSE,StringValue
705548,03LIC_1071,PID,OPHITP,,,
705549,03LIC_1071,PID,OPLOTP,,,
705528,03LIC_1071,PID,OPROCLM,,,
705481,03LIC_1071,PID,PVHIPR,HIGH,HIGH,StringValue
705502,03LIC_1071,PID,PVHITP,71.25,71.25,RealValue
705485,03LIC_1071,PID,PVLOPR,LOW,LOW,StringValue
705503,03LIC_1071,PID,PVLOTP,28.75,28.75,RealValue
705506,03LIC_1071,PID,PVROCNTP,,,


03LIC_1071 spot-check OK


## Phase D — Coverage analysis

For each control-action tag we determine:
- Was a DCS block found at all? (`NO BLOCK`)
- Which green params are applicable to its template? (`N/A` if not)
- Which applicable params are present vs empty (`MISSING`)
- Sentinel `1.79e308` is reported as `UNBOUNDED`

Relevance is also suffix-aware: for an `.OP` handle only the OP-family params are operationally relevant; for `.SP` handles, SP + PV families matter.

In [17]:
# ---- template applicability (empirical) --------------------------------------
# presence rate per (template, param) across ALL tags in the dump
all_blocks = df_csv.drop_duplicates(subset=['effective_tag','block_template'])
print('Total distinct blocks:', len(all_blocks))

template_counts = all_blocks['block_template'].value_counts()
print('\nTemplate counts:')
print(template_counts)

# presence per template (non-empty interpreted value, excluding UNBOUNDED sentinel counts as present)
present = (df_green_vals[df_green_vals['interpreted_value'] != '']
           .groupby(['template','ParameterName']).size()
           .reset_index(name='present'))
applicability = []
for template in df_green_vals['template'].unique():
    if pd.isna(template) or template == '':
        continue
    total = (all_blocks['block_template'] == template).sum()
    for param in green_params:
        p = present[(present['template']==template) & (present['ParameterName']==param)]['present'].sum()
        applicability.append({
            'template': template,
            'ParameterName': param,
            'n_blocks': total,
            'n_present': p,
            'presence_rate': p/total if total else np.nan
        })
df_applicability = pd.DataFrame(applicability)

# mark applicable if presence rate >= 0.5 (empirical threshold)
APPLICABILITY_THRESHOLD = 0.5
df_applicability['applicable'] = df_applicability['presence_rate'] >= APPLICABILITY_THRESHOLD

print('\nApplicability rates (sample):')
display(df_applicability.pivot(index='ParameterName', columns='template', values='presence_rate').fillna(0).head(20))

# AUTOMAN-specific check
print('\nAUTOMAN applicability:')
display(df_applicability[df_applicability['template']=='AUTOMAN'])

Total distinct blocks: 27678

Template counts:
block_template
CONTROLMODULE    13839
                 12667
CALCULTR           607
PID                244
DATAACQ             86
AUTOMAN             77
TOTALIZR            74
FLOWCOMP            28
GENLIN              14
SUMMER              12
ORSEL                9
SWITCH               9
RAMPSOAK             8
POSPROP              3
HILOAVG              1
Name: count, dtype: int64

Applicability rates (sample):


template,AUTOMAN,CALCULTR,DATAACQ,FLOWCOMP,GENLIN,HILOAVG,ORSEL,PID,POSPROP,RAMPSOAK,SUMMER,SWITCH,TOTALIZR
ParameterName,,,,,,,,,,,,,
$OPTOL,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.0
CTLACTN,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,1.000000,0.000000,0.0,0.000000,0.0,0.0
OPHITP,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.0
OPLOTP,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.111111,0.000000,0.000000,0.0,0.000000,0.0,0.0
OPROCLM,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.0
PVHIPR,0.0,1.000000,1.000000,1.000000,1.0,1.0,0.000000,0.995902,1.000000,0.0,0.666667,0.0,1.0
PVHITP,0.0,0.044481,0.151163,0.035714,0.0,0.0,0.000000,0.573770,0.333333,0.0,0.000000,0.0,0.0
PVLOPR,0.0,1.000000,1.000000,1.000000,1.0,1.0,0.000000,0.995902,1.000000,0.0,0.666667,0.0,1.0
PVLOTP,0.0,0.000000,0.174419,0.071429,0.0,0.0,0.000000,0.651639,0.333333,0.0,0.000000,0.0,0.0



AUTOMAN applicability:


,template,ParameterName,n_blocks,n_present,presence_rate,applicable
84,AUTOMAN,PVHITP,77,0,0.0,False
85,AUTOMAN,PVLOTP,77,0,0.0,False
86,AUTOMAN,PVHIPR,77,0,0.0,False
87,AUTOMAN,PVLOPR,77,0,0.0,False
88,AUTOMAN,OPHITP,77,0,0.0,False
89,AUTOMAN,OPLOTP,77,0,0.0,False
90,AUTOMAN,SPHILM,77,0,0.0,False
91,AUTOMAN,SPLOLM,77,0,0.0,False
92,AUTOMAN,CTLACTN,77,0,0.0,False
93,AUTOMAN,SPTOL,77,0,0.0,False


In [18]:
# ---- build coverage grid ------------------------------------------------------
# long table of latest value per (tag, param)
df_latest = (df_green_vals[df_green_vals['interpreted_value'] != '']
             .sort_values(['tag','ParameterName','raw_value'])
             .drop_duplicates(subset=['tag','ParameterName'], keep='last'))

# tags that exist in CSV (controllers + AI points)
csv_tags = set(df_csv['effective_tag'].dropna().unique())
print('Tags present in CSV dump:', len(csv_tags))
print('Source tags from actions:', len(source_tags))

# coverage per source tag x param
records = []
for tag in source_tags:
    tag_template = ''
    if tag in csv_tags:
        tmpl = df_green_vals[df_green_vals['tag']==tag]['template'].dropna()
        if len(tmpl):
            tag_template = tmpl.mode().iloc[0] if not tmpl.mode().empty else tmpl.iloc[0]
    rec = {'tag': tag, 'template': tag_template, 'block_found': tag in csv_tags}
    n_present = n_missing = n_na = 0
    for param in green_params:
        app = df_applicability[(df_applicability['template']==tag_template) &
                               (df_applicability['ParameterName']==param)]
        is_applicable = bool(app['applicable'].iloc[0]) if len(app) else False
        if tag not in csv_tags:
            cell = 'NO BLOCK'
        else:
            val_rows = df_latest[(df_latest['tag']==tag) & (df_latest['ParameterName']==param)]
            if len(val_rows):
                cell = val_rows['interpreted_value'].iloc[0]
                n_present += 1
            elif is_applicable:
                cell = 'MISSING'
                n_missing += 1
            else:
                cell = 'N/A'
                n_na += 1
        rec[param] = cell
    rec['n_present'] = n_present
    rec['n_missing'] = n_missing
    rec['n_na'] = n_na
    records.append(rec)

df_coverage = pd.DataFrame(records)
cols_front = ['tag','template','block_found','n_present','n_missing','n_na']
df_coverage = df_coverage[cols_front + [c for c in df_coverage.columns if c not in cols_front]]

print('Coverage grid shape:', df_coverage.shape)
print('Tags with NO BLOCK:', (~df_coverage['block_found']).sum())
display(df_coverage.head(20))

Tags present in CSV dump: 13839
Source tags from actions: 195
Coverage grid shape: (195, 20)
Tags with NO BLOCK: 58


,tag,template,block_found,n_present,n_missing,n_na,PVHITP,PVLOTP,PVHIPR,PVLOPR,OPHITP,OPLOTP,SPHILM,SPLOLM,CTLACTN,SPTOL,$OPTOL,OPROCLM,PVROCPTP,PVROCNTP
0,02EDPV_1052,,True,0,0,14,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
1,02EDPV_1094,,True,0,0,14,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
2,02FAX_1247,,False,0,0,0,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK
3,02FIC_1247,PID,True,6,1,7,MISSING,10535.0,NOACTION,LOW,N/A,N/A,52677,0,REVERSE,N/A,N/A,N/A,N/A,N/A
4,02HIC_1050,AUTOMAN,True,0,0,14,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
5,02HIC_1087,AUTOMAN,True,0,0,14,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
6,02KM_1139_ST,,False,0,0,0,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK
7,02LI_1041_MOS,,False,0,0,0,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK
8,02LI_1085_MOS,,False,0,0,0,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK
9,02PDC_1220_MAN,,False,0,0,0,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK,NO BLOCK


In [19]:
# ---- suffix-aware focus-handle deep-dive -------------------------------------
family_relevant = {
    'OP': OP_FAMILY,
    'SP': SP_FAMILY | PV_FAMILY,
}
focus_records = []
for handle in focus:
    tag, kind = handle.split('.')
    rel = family_relevant.get(kind, set()) & set(green_params)
    row = df_coverage[df_coverage['tag']==tag]
    if row.empty:
        focus_records.append({'handle':handle,'tag':tag,'kind':kind,'template':'NO BLOCK',
                              'relevant_params':sorted(rel),'present':[],'missing':[],'na':[],'unbounded':[]})
        continue
    template = row['template'].iloc[0]
    present=[]; missing=[]; na=[]; unbounded=[]
    for p in sorted(rel):
        v = row[p].iloc[0]
        if v == 'UNBOUNDED': unbounded.append(p)
        elif v == 'MISSING': missing.append(p)
        elif v == 'N/A': na.append(p)
        elif v == 'NO BLOCK': missing.append(f'{p}(NO BLOCK)')
        else: present.append(f'{p}={v}')
    focus_records.append({
        'handle': handle, 'tag': tag, 'kind': kind, 'template': template,
        'relevant_params': sorted(rel),
        'present': present, 'missing': missing, 'na': na, 'unbounded': unbounded,
        'n_relevant': len(rel), 'n_present': len(present), 'n_missing': len(missing),
        'n_na': len(na), 'n_unbounded': len(unbounded)
    })

df_focus = pd.DataFrame(focus_records)
# expand list columns to strings for Excel
df_focus['present_str'] = df_focus['present'].apply(lambda x: '; '.join(x))
df_focus['missing_str'] = df_focus['missing'].apply(lambda x: '; '.join(x))
df_focus['na_str']      = df_focus['na'].apply(lambda x: '; '.join(x))
df_focus['unbounded_str']= df_focus['unbounded'].apply(lambda x: '; '.join(x))
df_focus_export = df_focus[['handle','tag','kind','template','n_relevant','n_present','n_missing','n_na','n_unbounded',
                            'present_str','missing_str','na_str','unbounded_str']]
display(df_focus_export)

# assert focus tags found
for handle in focus:
    tag = handle.split('.')[0]
    assert tag in csv_tags, f'Focus tag {tag} not found in CSV dump'

,handle,tag,kind,template,n_relevant,n_present,n_missing,n_na,n_unbounded,present_str,missing_str,na_str,unbounded_str
0,03LIC_1071.OP,03LIC_1071,OP,PID,4,0,0,4,0,,,$OPTOL; OPHITP; OPLOTP; OPROCLM,
1,03PIC_1023.OP,03PIC_1023,OP,PID,4,0,0,4,0,,,$OPTOL; OPHITP; OPLOTP; OPROCLM,
2,03LIC_1034.SP,03LIC_1034,SP,PID,9,6,0,3,0,PVHIPR=HIGH; PVHITP=75.0; PVLOPR=HIGH; PVLOTP=...,,PVROCNTP; PVROCPTP; SPTOL,
3,03HIC_1151.OP,03HIC_1151,OP,AUTOMAN,4,0,0,4,0,,,$OPTOL; OPHITP; OPLOTP; OPROCLM,
4,03LIC_1016.OP,03LIC_1016,OP,PID,4,0,0,4,0,,,$OPTOL; OPHITP; OPLOTP; OPROCLM,


In [20]:
# ---- summary statistics --------------------------------------------------------
# per-template coverage
print('=== Per-template block counts ===')
print(df_coverage['template'].value_counts(dropna=False))

print('\n=== Overall param presence ===')
present_counts = {}
for p in green_params:
    present_counts[p] = (df_coverage[p].apply(lambda v: v not in ['MISSING','N/A','NO BLOCK',''])).sum()
df_param_summary = pd.DataFrame({
    'ParameterName': green_params,
    'family': [family_of(p) for p in green_params],
    'present_in_n_tags': [present_counts[p] for p in green_params],
    'missing_in_n_tags': [(df_coverage[p]=='MISSING').sum() for p in green_params],
    'na_in_n_tags': [(df_coverage[p]=='N/A').sum() for p in green_params],
    'no_block': [(df_coverage[p]=='NO BLOCK').sum() for p in green_params],
    'unbounded': [(df_coverage[p]=='UNBOUNDED').sum() for p in green_params],
})
df_param_summary['total_source_tags'] = len(source_tags)
display(df_param_summary.sort_values('present_in_n_tags', ascending=False))

print('\n=== NO BLOCK tags (real coverage gaps) ===')
no_block_tags = df_coverage.loc[~df_coverage['block_found'], 'tag'].tolist()
print(no_block_tags[:50])
print('count:', len(no_block_tags))

=== Per-template block counts ===
template
            152
PID          27
AUTOMAN      15
RAMPSOAK      1
Name: count, dtype: int64

=== Overall param presence ===


,ParameterName,family,present_in_n_tags,missing_in_n_tags,na_in_n_tags,no_block,unbounded,total_source_tags
3,PVLOPR,PV,34,0,103,58,0,195
2,PVHIPR,PV,34,0,103,58,0,195
7,SPLOLM,SP,27,0,110,58,0,195
8,CTLACTN,OTHER,27,0,110,58,0,195
6,SPHILM,SP,27,0,110,58,0,195
1,PVLOTP,PV,24,6,107,58,0,195
0,PVHITP,PV,19,11,107,58,0,195
4,OPHITP,OP,0,0,137,58,0,195
5,OPLOTP,OP,0,0,137,58,0,195
9,SPTOL,SP,0,0,137,58,0,195



=== NO BLOCK tags (real coverage gaps) ===
['02FAX_1247', '02KM_1139_ST', '02LI_1041_MOS', '02LI_1085_MOS', '02PDC_1220_MAN', '02XAX_1003', '02XAX_1004', '02XAX_1005', '02XAX_1007', '02XAX_1011', '02XAX_1012', '02XAX_1029_SEL', '02XAX_1090_SEL', '02XAX_1092_SEL', '02XAX_1093_SEL', '02XAX_1095_SEL', '02XAX_1143_RST', '02XAX_1220_AUT', '02XAX_1220_MAN', '03HS_3K101', '03LI_1033_MOS', '03LI_1035_MOS', '03LI_1086_MOS', '03LI_1128_MOS', '03LI_1174_MOS', '03LI_3151_MOS', '03PAX_1012', '03PDI_0007_MOS', '03PDI_0011_MOS', '03PDI_0016_MOS', '03PDI_0018_MOS', '03SI_3131_MOS', '03TI_0018_MOS', '03TI_0019_MOS', '03TI_0042A_MOS', '03TI_0043A_MOS', '03TI_0044A_MOS', '03TI_0045A_MOS', '03TI_3113_MOS', '03TI_3124_MOS', '03XAX_1001', '03XAX_1002', '03XAX_1013', '03XAX_1020', '03XAX_1024', '03XAX_1025', '03XAX_1027', '03XAX_1094', '03XAX_1220_RST', '03XAX_1220_SEL']
count: 58


## Phase E — Export results workbook

In [21]:
# ---- build How_to_read text --------------------------------------------------
readme_lines = [
    "DCS constraints extraction for 03LIC_1071 PVLO control-action tags.",
    "",
    "Sheets:",
    "  green_parameters      : the green-flagged DCS Parameter List values from the Excel,",
    "                          exploded into individual ParameterName entries with their family.",
    "  constraint_values_long: all non-empty green-parameter values found in Tags_Parameters.csv.",
    "  coverage_grid         : one row per Source tag from the 1071 control_actions sheet; one",
    "                          column per green parameter. Cell values:",
    "                            <value>   = extracted DCS value (coalesced from Integer/Real/String)",
    "                            UNBOUNDED = sentinel 1.79e308 (limit disabled / unbounded)",
    "                            N/A       = parameter not applicable to this tag's DCS template",
    "                            MISSING   = applicable to template but value is empty",
    "                            NO BLOCK  = the Source tag has no matching block in Tags_Parameters.csv",
    "  focus_handles         : deep-dive on the 5 requested handles (03LIC_1071.OP, 03PIC_1023.OP,",
    "                          03LIC_1034.SP, 03HIC_1151.OP, 03LIC_1016.OP). Relevance is suffix-aware:",
    "                          .OP -> OP-family limits; .SP -> SP + PV family limits.",
    "  coverage_summary      : per-template block counts and per-parameter presence/missing/N-A tallies.",
    "",
    "Key caveat: DCS numeric limits live on function blocks where StrategyName is blank and the tag",
    "is named only by the NAME row (TemplateName PID or AUTOMAN). AI points are keyed by StrategyName.",
    "The segmentation forward-fills the NAME value across each controller block.",
]
df_readme = pd.DataFrame({'How to read this file': readme_lines})

# ---- write workbook ----------------------------------------------------------
with pd.ExcelWriter(OUT_XLS, engine='openpyxl') as writer:
    df_green.to_excel(writer, sheet_name='green_parameters', index=False)
    df_green_vals.to_excel(writer, sheet_name='constraint_values_long', index=False)
    df_coverage.to_excel(writer, sheet_name='coverage_grid', index=False)
    df_focus_export.to_excel(writer, sheet_name='focus_handles', index=False)
    df_param_summary.to_excel(writer, sheet_name='coverage_summary', index=False)
    df_applicability.to_excel(writer, sheet_name='applicability_rates', index=False)
    df_readme.to_excel(writer, sheet_name='How_to_read', index=False)

print('Saved ->', OUT_XLS)

Saved -> /home/h604827/ControlActions/RESULTS/dcs_constraints_1071/dcs_constraints_coverage_1071.xlsx


In [22]:
# ---- final summary for the user ----------------------------------------------
print(f"Source tags from 1071 control_actions: {len(source_tags)}")
print(f"Tags found in Tags_Parameters.csv:     {df_coverage['block_found'].sum()}")
print(f"Tags with NO BLOCK:                    {(~df_coverage['block_found']).sum()}")
print(f"Green parameters extracted:            {len(green_params)}")
print()
print("Focus-handle coverage:")
for _, r in df_focus_export.iterrows():
    print(f"  {r['handle']:22s} template={r['template']:8s} relevant={r['n_relevant']:2d} "
          f"present={r['n_present']:2d} missing={r['n_missing']:2d} N/A={r['n_na']:2d} unbounded={r['n_unbounded']:2d}")
print()
print("Most-frequently MISSING applicable params:")
missing_counts = df_param_summary[df_param_summary['missing_in_n_tags']>0].sort_values('missing_in_n_tags', ascending=False)
display(missing_counts[['ParameterName','family','missing_in_n_tags','na_in_n_tags','no_block']].head(10))

Source tags from 1071 control_actions: 195
Tags found in Tags_Parameters.csv:     137
Tags with NO BLOCK:                    58
Green parameters extracted:            14

Focus-handle coverage:
  03LIC_1071.OP          template=PID      relevant= 4 present= 0 missing= 0 N/A= 4 unbounded= 0
  03PIC_1023.OP          template=PID      relevant= 4 present= 0 missing= 0 N/A= 4 unbounded= 0
  03LIC_1034.SP          template=PID      relevant= 9 present= 6 missing= 0 N/A= 3 unbounded= 0
  03HIC_1151.OP          template=AUTOMAN  relevant= 4 present= 0 missing= 0 N/A= 4 unbounded= 0
  03LIC_1016.OP          template=PID      relevant= 4 present= 0 missing= 0 N/A= 4 unbounded= 0

Most-frequently MISSING applicable params:


,ParameterName,family,missing_in_n_tags,na_in_n_tags,no_block
0,PVHITP,PV,11,107,58
1,PVLOTP,PV,6,107,58
